# Dashboard 2: Product Portfolio & "Bleeding SKUs" - Exploratory Data Analysis (EDA)

This notebook conducts a granular SKU-level analysis to evaluate product performance, identify "false positives" (vanity metrics vs. sanity net margins), and isolate unprofitable items (Bleeding SKUs).

## 1. Setup & Visual Theme Initialization
Load libraries and establish a polished graphical design system using Seaborn and Matplotlib.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Configure premium styling
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['axes.edgecolor'] = '#e2e8f0'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.color'] = '#f1f5f9'
print("Libraries loaded & styling configured.")

## 2. Load Gold Layer Output Data
Read the fact and dimension files generated by the Gold Layer pipeline and merge them into a unified frame.

In [ ]:
prod_path = "Medallion Architecture/Gold Layer/dim_product.csv"
sales_path = "Medallion Architecture/Gold Layer/fact_sales.csv"

if not os.path.exists(prod_path) or not os.path.exists(sales_path):
    raise FileNotFoundError("Gold layer output files not found. Ensure they are present in Medallion Architecture/Gold Layer/")

df_prod = pd.read_csv(prod_path)
df_sales = pd.read_csv(sales_path)

# Merge fact sales table with product dimensions on Product_Key
df_merged = pd.merge(df_sales, df_prod, on="Product_Key", how="left")
print(f"Loaded {len(df_merged)} transaction records across {df_merged['SKU'].nunique()} unique SKUs.")
df_merged.head()

## 3. Dashboard 2 KPI Cards Calculation
Derive the key top-level metrics for the SKU-level performance portfolio:
* **Total Distinct Products**
* **Top Product Revenue (SKU & Amount)**
* **Average Revenue per Product**
* **Total COGS**

In [ ]:
# Aggregate data at SKU level
sku_summary = df_merged.groupby("SKU").agg(
    Total_Revenue=("Revenue", "sum"),
    Total_Units=("Units", "sum"),
    Total_COGS=("COGS", lambda x: x.abs().sum()),
    Total_FBA=("FBA Fees", lambda x: x.abs().sum()),
    Total_PPC=("PPC Cost", lambda x: x.abs().sum()),
    Total_Taxes=("Taxes", lambda x: x.abs().sum()),
    Total_Refunds=("Refunded", lambda x: x.abs().sum()),
    Total_Net_Profit=("Net Profit", "sum")
).reset_index()

sku_summary["Net_Margin_%"] = (sku_summary["Total_Net_Profit"] / sku_summary["Total_Revenue"]) * 100
sku_summary["TACoS_%"] = (sku_summary["Total_PPC"] / sku_summary["Total_Revenue"]) * 100

# Calculate KPIs
distinct_products = sku_summary["SKU"].nunique()
top_product_idx = sku_summary["Total_Revenue"].idxmax()
top_product_sku = sku_summary.loc[top_product_idx, "SKU"]
top_product_revenue = sku_summary.loc[top_product_idx, "Total_Revenue"]

total_revenue = sku_summary["Total_Revenue"].sum()
avg_revenue_per_product = total_revenue / distinct_products
total_cogs = sku_summary["Total_COGS"].sum()

print("="*50)
print("DASHBOARD 2 KPI CARDS")
print("="*50)
print(f"Total Distinct Products:       {distinct_products:,}")
print(f"Top Product Revenue:           ${top_product_revenue:,.2f} ({top_product_sku})")
print(f"Average Revenue per Product:   ${avg_revenue_per_product:,.2f}")
print(f"Total COGS (Absolute Cost):    ${total_cogs:,.2f}")
print("="*50)

## 4. Visual Analysis & Charts

### Chart 1: Top 10 Products by Revenue (Horizontal Bar Chart)
Identify the primary revenue engines of the e-commerce portfolio.

In [ ]:
top10_rev = sku_summary.nlargest(10, "Total_Revenue")

plt.figure(figsize=(12, 6.5))
sns.barplot(
    data=top10_rev,
    y="SKU",
    x="Total_Revenue",
    palette="viridis",
    hue="SKU",
    legend=False
)

# Formatting labels
for index, value in enumerate(top10_rev["Total_Revenue"]):
    plt.text(value + (top10_rev["Total_Revenue"].max() * 0.005), index, f"${value:,.0f}", 
             va='center', fontsize=9, fontweight='bold', color='#334155')

plt.title('Top 10 Products by Revenue', fontsize=14, fontweight='bold', pad=15, color='#1e293b')
plt.xlabel('Total Revenue ($)', fontsize=12, labelpad=10, color='#475569')
plt.ylabel('Product SKU', fontsize=12, color='#475569')
plt.tight_layout()
plt.show()

### Chart 2: Profit Pareto 80/20 Rule (Pareto Chart)
Analyze the concentration of Net Profits across products to verify if 80% of net profits are driven by 20% of SKUs.

In [ ]:
# Sort SKUs by net profit descending
pareto_df = sku_summary.sort_values(by="Total_Net_Profit", ascending=False).reset_index(drop=True)
pareto_df["Cumulative_Profit"] = pareto_df["Total_Net_Profit"].cumsum()
overall_profit = pareto_df["Total_Net_Profit"].sum()

# We normalize against positive profits or overall profit. 
# If overall_profit is used, we plot cumulative contribution.
pareto_df["Cumulative_Profit_%"] = (pareto_df["Cumulative_Profit"] / overall_profit) * 100

fig, ax1 = plt.subplots(figsize=(13, 6))

# Individual profit bars
ax1.bar(pareto_df.index, pareto_df["Total_Net_Profit"], color="#0284c7", alpha=0.8, label="Net Profit")
ax1.set_xlabel("SKUs (ordered by descending profitability)", fontsize=12, labelpad=10, color="#475569")
ax1.set_ylabel("Net Profit ($)", color="#0284c7", fontsize=12)
ax1.tick_params(axis="y", labelcolor="#0284c7")
ax1.set_xlim(-1, len(pareto_df))

# Cumulative percentage curve
ax2 = ax1.twinx()
ax2.plot(pareto_df.index, pareto_df["Cumulative_Profit_%"], color="#e11d48", linewidth=2.5, marker="o", markersize=3, label="Cumulative Profit %")
ax2.set_ylabel("Cumulative Profit %", color="#e11d48", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#e11d48")
ax2.axhline(80, color="#64748b", linestyle="--", linewidth=1.5, alpha=0.8, label="80% Profit line")

plt.title("Profit Pareto 80/20 Distribution Chart", fontsize=14, fontweight="bold", pad=15, color="#1e293b")
fig.tight_layout()
plt.show()

### Chart 3: The "Vanity vs. Sanity" Matrix (Scatter Plot)
Compare top-line **Revenue** (Vanity) against **Net Margin %** (Sanity). Bubble size represents **Total Units Sold**, and color maps to **Total Net Profit**.

*This chart isolates high-revenue, low-margin products (False Positives) and low-revenue, high-margin opportunities (Hidden Gems).*

In [ ]:
plt.figure(figsize=(12, 7.5))

# Size mapping normalized for readability
max_units = sku_summary["Total_Units"].max()
size_scale = (sku_summary["Total_Units"] / max_units) * 800 + 80

# Color mapping based on Net Profit
scatter = plt.scatter(
    sku_summary["Total_Revenue"],
    sku_summary["Net_Margin_%"],
    s=size_scale,
    c=sku_summary["Total_Net_Profit"],
    cmap="RdYlGn",
    alpha=0.8,
    edgecolors="#475569",
    linewidths=0.8
)

# Label outliers and notable SKUs
for i, row in sku_summary.iterrows():
    # Label if SKU is in top 15% revenue or has negative margin
    if row["Total_Revenue"] > sku_summary["Total_Revenue"].quantile(0.8) or row["Net_Margin_%"] < 0:
        clean_label = row["SKU"].replace("UK_Aava_", "").replace("DE_Aava_", "").replace("FR_Aava_", "").replace("IT_Aava_", "").replace("ES_Aava_", "")
        plt.annotate(
            clean_label,
            (row["Total_Revenue"], row["Net_Margin_%"]),
            xytext=(6, 4),
            textcoords="offset points",
            fontsize=8.5,
            fontweight="semibold",
            color="#1e293b",
            alpha=0.9
        )

plt.axhline(0, color="#ef4444", linestyle="--", linewidth=1.5, alpha=0.8)
plt.title('The "Vanity vs. Sanity" Portfolio Matrix', fontsize=14, fontweight='bold', pad=15, color='#1e293b')
plt.xlabel('Total Revenue ($)', fontsize=12, color='#475569')
plt.ylabel('Net Margin (%)', fontsize=12, color='#475569')

# Add legend colorbar
cbar = plt.colorbar(scatter)
cbar.set_label('Total Net Profit ($)', rotation=270, labelpad=15, color='#475569')
plt.tight_layout()
plt.show()

### Chart 4: Margin Erosion by SKU (100% Stacked Bar)
For the **Top 10 Products by Revenue**, show how costs (COGS, FBA Fees, PPC Costs, Taxes) erode the gross revenue, displaying the remaining Net Margin %.

In [ ]:
top10_skus = sku_summary.nlargest(10, "Total_Revenue").copy()

# Calculate percentages relative to Total Revenue
top10_skus["COGS_%"] = (top10_skus["Total_COGS"] / top10_skus["Total_Revenue"]) * 100
top10_skus["FBA_%"] = (top10_skus["Total_FBA"] / top10_skus["Total_Revenue"]) * 100
top10_skus["PPC_%"] = (top10_skus["Total_PPC"] / top10_skus["Total_Revenue"]) * 100
top10_skus["Taxes_%"] = (top10_skus["Total_Taxes"] / top10_skus["Total_Revenue"]) * 100

# Stacking values for 100% chart visualization
y_pos = np.arange(len(top10_skus))
labels = top10_skus["SKU"].values

cogs = top10_skus["COGS_%"].values
fba = top10_skus["FBA_%"].values
ppc = top10_skus["PPC_%"].values
taxes = top10_skus["Taxes_%"].values
margin = top10_skus["Net_Margin_%"].values

plt.figure(figsize=(13, 7.5))

plt.barh(y_pos, cogs, label="COGS %", color="#f87171", alpha=0.9)
plt.barh(y_pos, fba, left=cogs, label="FBA Fees %", color="#fbbf24", alpha=0.9)
plt.barh(y_pos, ppc, left=cogs+fba, label="PPC Cost %", color="#f97316", alpha=0.9)
plt.barh(y_pos, taxes, left=cogs+fba+ppc, label="Taxes %", color="#60a5fa", alpha=0.9)
plt.barh(y_pos, margin, left=cogs+fba+ppc+taxes, label="Net Margin %", color="#34d399", alpha=0.9)

plt.yticks(y_pos, labels, fontsize=10)
plt.xlabel("Percentage of Total Revenue (%)", fontsize=12, color="#475569")
plt.title("Margin Erosion Breakdown (Top 10 SKUs by Revenue)", fontsize=14, fontweight="bold", pad=15, color="#1e293b")
plt.legend(loc="lower center", bbox_to_anchor=(0.5, -0.15), ncol=5, frameon=True, facecolor="#f8fafc", edgecolor="#cbd5e1")
plt.tight_layout()
plt.show()

### Chart 5: Bleeding SKUs Alert (Data Table)
Identify products with negative Net Profit. Includes critical metrics: Net Margin %, TACoS % (Total Ad Cost of Sales), Refund Rate %, and FBA Fees.

In [ ]:
# Aggregate refund metrics
sku_refunds = df_merged.groupby("SKU").agg(
    Total_Units=("Units", "sum"),
    Total_Refunds=("Refunded", "sum")
).reset_index()

sku_summary = pd.merge(sku_summary, sku_refunds[["SKU", "Total_Refunds"]], on="SKU")
sku_summary["Refund_Rate_%"] = (sku_summary["Total_Refunds"] / sku_summary["Total_Units"]) * 100

# Filter for Bleeding SKUs (Net Profit < 0)
bleeding_skus = sku_summary[sku_summary["Total_Net_Profit"] < 0].copy()

# Select alert fields
alert_table = bleeding_skus[[
    "SKU", "Total_Revenue", "Total_Net_Profit", "Net_Margin_%", "TACoS_%", "Refund_Rate_%", "Total_FBA"
]].sort_values(by="Total_Net_Profit")

print("="*100)
print("                           BLEEDING SKUS ALERT: CRITICAL CASH DRAINS")
print("="*100)
if len(alert_table) == 0:
    print("All products are generating positive net profit. No bleeding SKUs detected.")
else:
    print(alert_table.to_string(index=False, formatters={
        'Total_Revenue': '${:,.2f}'.format,
        'Total_Net_Profit': '${:,.2f}'.format,
        'Net_Margin_%': '{:.2f}%'.format,
        'TACoS_%': '{:.2f}%'.format,
        'Refund_Rate_%': '{:.2f}%'.format,
        'Total_FBA': '${:,.2f}'.format
    }))
print("="*100)

## 5. Strategic Recommendations
Based on the analysis:
1. **Prune/Reprice Bleeding SKUs:** Examine SKUs in the Bleeding SKU alert list. If TACoS is extremely high, freeze or optimize PPC bids. If Refund Rate is high (>10%), audit physical quality control.
2. **Defend Star Performers:** Secure stock levels for top revenue earners with strong margins.
3. **Address Margin Erosion:** Identify where FBA fees or PPC spend are disproportionately high and optimize packaging dimensions/weight.